# Download a folder from an Azure ML job's outputs into `./results/`

Set `JOB_NAME` to the job (Run ID, e.g. `epic_river_xyz123`), `SOURCE_SUBDIR`
to the path inside the job's `outputs/` you want (e.g. `adapter` or
`moonshine-tiny-adalora`), and `TARGET_NAME` to the name under `./results/`
where it should land.

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("RESOURCE_GROUP"),
    workspace_name=os.getenv("WORKSPACE"),
)

In [ ]:
# --- configure here ---
JOB_NAME = "<run-id-from-studio>"            # e.g. 'epic_river_xyz123'
SOURCE_SUBDIR = "adapter"                     # path inside the job's outputs/
TARGET_NAME = SOURCE_SUBDIR                   # folder name under ./results/
OVERWRITE = True

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
target_path = RESULTS_DIR / TARGET_NAME
if target_path.exists():
    if not OVERWRITE:
        raise FileExistsError(f"{target_path} already exists (set OVERWRITE=True to replace)")
    shutil.rmtree(target_path)

with tempfile.TemporaryDirectory(prefix="azureml-dl-") as tmp:
    tmp_path = Path(tmp)
    print(f"Downloading job '{JOB_NAME}' outputs into {tmp_path} ...")
    ml_client.jobs.download(name=JOB_NAME, download_path=str(tmp_path), all=True)

    # Azure ML places the default command-output under `named-outputs/default/outputs/`
    # or directly under `outputs/` depending on SDK version. Try both.
    candidates = [
        tmp_path / "named-outputs" / "default" / SOURCE_SUBDIR,
        tmp_path / "outputs" / SOURCE_SUBDIR,
        tmp_path / SOURCE_SUBDIR,
    ]
    source_path = next((c for c in candidates if c.exists()), None)

    if source_path is None:
        print("Could not locate source subdir. Downloaded tree:")
        for p in sorted(tmp_path.rglob("*")):
            rel = p.relative_to(tmp_path)
            marker = "/" if p.is_dir() else ""
            print(f"  {rel}{marker}")
        raise FileNotFoundError(
            f"'{SOURCE_SUBDIR}' not found under any known outputs location."
        )

    print(f"Copying {source_path} -> {target_path}")
    shutil.copytree(source_path, target_path)

print(f"\nDone. Contents of {target_path}:")
for p in sorted(target_path.rglob("*")):
    rel = p.relative_to(target_path)
    marker = "/" if p.is_dir() else f" ({p.stat().st_size} bytes)"
    print(f"  {rel}{marker}")